# Text Generation using Transformers

In [1]:
import numpy as np 
import pandas as pd 
import os
import transformers 
from transformers import AutoTokenizer
import torch 
from torch.utils.data import DataLoader 
from datasets import load_dataset,DatasetDict
from transformers import AutoTokenizer
from transformers import DataCollatorForLanguageModeling
from accelerate import Accelerator
from transformers import get_scheduler
from tqdm.notebook import tqdm

In [2]:
train_ds = load_dataset("huggingface-course/codeparrot-ds-train",split='train')
test_ds = load_dataset("huggingface-course/codeparrot-ds-valid",split='validation')
raw_dataset = DatasetDict(
    {
        "train" : train_ds.shuffle().select(range(50000)),
        "validate" : test_ds.shuffle().select(range(1000))
    }
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


codeparrot-ds-train.jsonl:   0%|          | 0.00/8.25G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

codeparrot-ds-valid.jsonl:   0%|          | 0.00/46.1M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/3322 [00:00<?, ? examples/s]

In [3]:
print("Data Overview")
print("-"*100)
print(raw_dataset['train'][1]['content'][:200])
print("-"*100)

Data Overview
----------------------------------------------------------------------------------------------------
"""
To run this, you'll need to have installed.

  * scikit-learn

Does two benchmarks

First, we fix a training set, increase the number of
samples to classify and plot number of classified samples a
----------------------------------------------------------------------------------------------------


In [4]:
checkpoint = ''
tokenizer = AutoTokenizer.from_pretrained("huggingface-course/code-search-net-tokenizer")

tokenizer_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

**Preprocessing the dataset**

In [5]:
context_length = 128
def tokenize_function(example):
    outputs = tokenizer(
        example['content'],
        truncation = True,
        max_length = context_length,
        return_overflowing_tokens = True,
        return_length = True
    )
    input_batch = []
    for length,input_ids in zip(outputs['length'],outputs['input_ids']):
        if length == context_length:
            input_batch.append(input_ids)
    return {"input_ids":input_batch}

In [6]:
tokenized_dataset = raw_dataset.map(tokenize_function,batched=True,remove_columns=raw_dataset['train'].column_names)
tokenized_dataset

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 1373207
    })
    validate: Dataset({
        features: ['input_ids'],
        num_rows: 29739
    })
})

In [7]:
from transformers import GPT2LMHeadModel,AutoConfig

config = AutoConfig.from_pretrained(
    pretrained_model_name_or_path = "gpt2",
    vocab_size = len(tokenizer),
    n_ctx = context_length,
    bos_token_id = tokenizer.bos_token_id,
    eos_token_id = tokenizer.bos_token_id
)
model = GPT2LMHeadModel(config)
model_size = sum(t.numel() for t in model.parameters())
print(f"GPT-2 parameters : {model_size/1000**2:.1f}M Paramters")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

GPT-2 parameters : 124.2M Paramters


In [22]:
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)

In [23]:
def keytoken_weighted_loss(inputs, logits, keytoken_ids, alpha=1.0):
    shift_labels = inputs[..., 1:].contiguous()
    shift_logits = logits[..., :-1, :].contiguous()

    loss_fct = torch.nn.CrossEntropyLoss(reduce=False)
    loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

    loss_per_sample = loss.view(shift_logits.size(0), shift_logits.size(1)).mean(axis=1)
    # Calculate and scale weighting
    weights = torch.stack([(inputs == kt).float() for kt in keytoken_ids]).sum(
        axis=[0, 2]
    )
    weights = alpha * (1.0 + weights)

    weighted_loss = (loss_per_sample * weights).mean()
    return weighted_loss

In [24]:
weight_decay = 0.1
def get_grouped_params(model, no_decay=["bias", "LayerNorm.weight"]):
    params_with_wd, params_without_wd = [], []
    for n, p in model.named_parameters():
        if any(nd in n for nd in no_decay):
            params_without_wd.append(p)
        else:
            params_with_wd.append(p)
    return [
        {"params": params_with_wd, "weight_decay": weight_decay},
        {"params": params_without_wd, "weight_decay": 0.0},
    ]

In [25]:
def evaluate():
    model.eval()
    losses = []
    for step, batch in enumerate(Eval_dataloader):
        with torch.no_grad():
            outputs = model(batch["input_ids"], labels=batch["input_ids"])

        losses.append(accelerator.gather(outputs.loss))
    loss = torch.mean(torch.cat(losses))
    try:
        perplexity = torch.exp(loss)
    except OverflowError:
        perplexity = float("inf")
    return loss.item(), perplexity.item()

In [26]:
Batch_size = 16
tokenized_dataset.set_format('torch')
Train_dataloader = DataLoader(
    tokenized_dataset['train'],
    shuffle=True,
    batch_size=Batch_size,
    collate_fn = data_collator,
)

Eval_dataloader = DataLoader(
    tokenized_dataset['validate'],
    shuffle=False,
    batch_size = Batch_size,
    collate_fn = data_collator,
)

In [27]:
Epochs = 1
optimizer = torch.optim.AdamW(get_grouped_params(model),lr=0.0002)
num_update_step_per_epoch = len(Train_dataloader)
num_train_steps = Epochs * num_update_step_per_epoch

lr_scheduler = get_scheduler(
    "linear",
    optimizer = optimizer,
    num_training_steps = num_train_steps,
    num_warmup_steps = 1
)

accelerator = Accelerator()
model,optimizer,Train_dataloader,Eval_dataloader = accelerator.prepare(model,optimizer,Train_dataloader,Eval_dataloader)

In [28]:
keytoken_ids = []
for keyword in [
    "plt",
    "pd",
    "sk",
    "fit",
    "predict",
    " plt",
    " pd",
    " sk",
    " fit",
    " predict",
    "testtest",
]:
    ids = tokenizer([keyword]).input_ids[0]
    if len(ids) == 1:
        keytoken_ids.append(ids[0])
    else:
        print(f"Keyword has not single token: {keyword}")
print(keytoken_ids)

Keyword has not single token: testtest
[8436, 4289, 1201, 2770, 5431, 2564, 2604, 2110, 2872, 4969]


In [ ]:
gradient_acc_step = 8
eval_steps = 5_000

model.train()
completed_steps = 0
for epoch in range(Epochs):
    for step,batch in tqdm(enumerate(Train_dataloader,start=1),total=num_train_steps):
        logits = model(batch['input_ids']).logits
        loss = keytoken_weighted_loss(batch["input_ids"], logits, keytoken_ids)
        if step % 100 ==0:
            accelerator.print(
                {
                    "steps": completed_steps,
                    "loss/train": loss.item() * gradient_acc_step,
                }
            )
        loss = loss / gradient_acc_step
        accelerator.backward(loss)
        if (step % gradient_acc_step == 0):
            accelerator.clip_grad_norm_(model.parameters(),1.0)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            completed_steps = completed_steps + 1 

        if (step % (eval_steps*gradient_acc_step)) == 0:
            eval_loss , perplexity = evaluate()
            accelerator.print({"loss/eval": eval_loss, "perplexity": perplexity})
            model.train()
            accelerator.wait_for_everyone()
            unwrapped_model = accelerator.unwrap_model(model)
            unwrapped_model.save_pretrained("/kaggle/working/Models/", save_function=accelerator.save)

# Inference 

In [ ]:
text = input("Enter text : ")
tokenized_text = tokenizer(text)

with torch.no_grad():
    output = model(tokenizer_text['input_ids'])
print(f"Generated text : {output}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated text : write code for ploting a graph in python. The code below will be used to generate a plot using the plot() function.

import graph from arc import Plotplot import LinearPlot import matrix import random def plot ( self, x ): """ Plot the x axis. """ plot ( self, x ) return plot ( x, z )

Example: plot(x, y): Plot a x axis.

" # plot plot (x, y) plot (x, y) plot (x, y) plot (x, y) plot (x, y) plot (x, y) plot (y) plot (y, z) plot (x, y) plot (x, z) plot (x, y) plot (x, z) plot (x, z) plot (x, y) plot (x, z) plot (x, y) plot (x, z) plot (x, z) plot (x, y) plot (x, z) plot (x, y) plot (x, z) plot (x, y) plot (x, z) plot (x, y) plot (x, z) plot (x, y) plot (x, z) plot (x, y) plot (x,
